### Advanced Python: Software Design Patterns

#### Day 6: Odds and Ends
  - Using metaclasses
  - Partials
  - Single Dispatch generic function

#### Capstone Project: `LogAnalyzer`

In [9]:
class User:
    pass

u = User()
print(type(u))

a = 100 # a = int(100)
print(type(a))

print(type(User))
print(type(int))

Car = type('Car', (), {})
print(Car)

<class '__main__.User'>
<class 'int'>
<class 'type'>
<class 'type'>
<class '__main__.Car'>


In [11]:
u.__class__.__class__

type

In [14]:
u.__class__.__base__

object

In [26]:
class Vehicle(type):
    pass

print(Vehicle)
print(Vehicle.__base__)

Bike = Vehicle('Bike', (), {})
Truck = Vehicle('Truck', (), {})
print(Bike, Truck)

t = Truck()
b = Bike()
print(t, b)
print(t.__class__.__class__.__class__)

<class '__main__.Vehicle'>
<class 'type'>
<class '__main__.Bike'> <class '__main__.Truck'>
<__main__.Truck object at 0x1038a0d40> <__main__.Bike object at 0x10385f8c0>
<class 'type'>


In [28]:
class Vehicle(type):
    pass

class Car(metaclass=Vehicle):
    pass

class Bike(metaclass=Vehicle):
    pass

class User:
    pass

print(Car.__class__)
print(Bike.__class__)
print(User.__class__)

<class '__main__.Vehicle'>
<class '__main__.Vehicle'>
<class 'type'>


In [30]:
def create_vehicle_class(name, bases=(), attrs={}):
    print(f"Creating vehicle class: {name=}, {bases=}, {attrs=}")

class Car(metaclass=create_vehicle_class):
    pass

print(Car)

Creating vehicle class: name='Car', bases=(), attrs={'__module__': '__main__', '__qualname__': 'Car'}
None


In [16]:
class Student(User): pass

Student.__base__.__base__

object

In [17]:
class User: pass
class User(object): pass

In [ ]:
class Parser:
    registry = {}

    def __new__(cls, source):   
        ext = source.split('.')[-1]
        parser_class = cls.get_parser(ext)
        instance = super().__new__(parser_class)
        return instance

    @classmethod
    def register(cls, format_name):
        def decorator(subclass):
            cls.registry[format_name] = subclass
            return subclass
        return decorator

    @classmethod
    def get_parser(cls, format_name):
        parser_class = cls.registry.get(format_name)
        if not parser_class:
            raise ValueError(f"No parser registered for format: {format_name}")
        return parser_class

    def __init__(self, source):
        self.source = source
    
    def parse(self):
        print(f"Parsing source: {self.source}")

#--------------------------------------------
@Parser.register('json')  # Explicit is better than implicit
class JSONParser(Parser):
    def parse(self):
        print(f"Parsing JSON source: {self.source}")

@Parser.register('xml')
class XMLParser(Parser):
    def parse(self):
        print(f"Parsing XML source: {self.source}")

@Parser.register('csv')
class CSVParser(Parser):
    def parse(self):
        print(f"Parsing CSV source: {self.source}")

#-----------------------------------------

p1 = Parser("testfile.json") # Return an instance of JSONParser
p2 = Parser("testfile.xml") # Return an instance of XMLParser
p1.parse()
p2.parse()

Parsing JSON source: testfile.json
Parsing XML source: testfile.xml


In [48]:
class Parser:

    @classmethod
    def register(cls, format_name):
        def decorator(subclass):
            setattr(cls, format_name, subclass)
            #cls.__dict__[format_name] = subclass
            return subclass
        return decorator

    def __init__(self, source):
        self.source = source
    
    def parse(self):
        print(f"Parsing source: {self.source}")

#--------------------------------------------
@Parser.register('json')
class JSONParser(Parser):
    def parse(self):
        print(f"Parsing JSON source: {self.source}")

@Parser.register('xml')
class XMLParser(Parser):
    def parse(self):
        print(f"Parsing XML source: {self.source}")

@Parser.register('csv')
class CSVParser(Parser):
    def parse(self):
        print(f"Parsing CSV source: {self.source}")

#-----------------------------------------

filename = "testdata.json"
ext = filename.split('.')[-1]


json_parser = getattr(Parser, ext)('data.json')
print(json_parser)


In [63]:
class Parser(type):
    registry = {}
    def __new__(cls, name, bases, attrs):
        new_class = super().__new__(cls, name, bases, attrs)
        format = attrs.get('format')
        if format:
            cls.registry[format] = new_class
        else:
            raise ValueError("Subclasses must define a 'format' attribute")
        return new_class
    
    @classmethod
    def get_parser(cls, source):
        cls.source = source
        format = source.split('.')[-1]
        parser_class = cls.registry.get(format)
        if not parser_class:
            raise ValueError(f"No parser registered for format: {format}")
        return parser_class(source)
        

class JSONParser(metaclass=Parser):
    format = 'json'
    def __init__(self, source):
        self.source = source

class XMLParser(metaclass=Parser):
    format = "xml"


Parser.registry
Parser.get_parser("data.json")

In [ ]:
class SingletonMeta(type):   # I would generally avoid this design
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
        return cls._instances[cls]
    
class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self):
        self.connection = "Database Connection Established"


db1 = DatabaseConnection()
db2 = DatabaseConnection()
print(db1 is db2)  # True

db1()

True


TypeError: 'DatabaseConnection' object is not callable

In [75]:
class Singleton:
    _instances = {}
    def __new__(cls, *args, **kwargs):
        if cls not in cls._instances:
            instance = super().__new__(cls)
            cls._instances[cls] = instance
        return cls._instances[cls]
    
class ConfigManager(Singleton):
    def __init__(self):
        self.config = {}
        print("ConfigManager initialized")

config1 = ConfigManager()
config2 = ConfigManager()
print(config1 is config2)  # True

ConfigManager initialized
ConfigManager initialized
True


In [ ]:
def singleton(cls):   # This is the best recommended approach
    instances = {}
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class DatabaseConnection:
    def __init__(self, host):
        self.host = host
        print(f"Connecting to database at {host}")

In [72]:
Car = type("Car", (), {})

dir(Car)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__']

In [ ]:
class Parser(type):  # Strategy Pattern (registry) using metaclass
    registry = {}
    def __new__(cls, name, bases, attrs):
        new_class = super().__new__(cls, name, bases, attrs)
        format = attrs.get('format')
        if format:
            cls.registry[format] = new_class
        else:
            raise ValueError("Subclasses must define a 'format' attribute")
        return new_class
    
    @classmethod
    def get_parser(cls, source):
        cls.source = source
        format = source.split('.')[-1]
        parser_class = cls.registry.get(format)
        if not parser_class:
            raise ValueError(f"No parser registered for format: {format}")
        return parser_class(source)
        

class JSONParser(metaclass=Parser):
    format = 'json'
    def __init__(self, source):
        self.source = source

class XMLParser(metaclass=Parser):
    format = "xml"


Parser.registry
Parser.get_parser("data.json")

In [80]:
class Parser:   # Use this approach over metaclasses
    registry = {}

    def __init_subclass__(cls):
        print(f"{cls=}")
        super().__init_subclass__()
        format = getattr(cls, 'format')
        if format:
            cls.registry[format] = cls
        else:
            raise ValueError("Subclasses must define a 'format' attribute")

class JSONParser(Parser):
    format = 'json'

class XMLParser(Parser):
    format = "xml"
    pass

Parser.registry

cls=<class '__main__.JSONParser'>
cls=<class '__main__.XMLParser'>


{'json': __main__.JSONParser, 'xml': __main__.XMLParser}

In [82]:
def partial(fn, *args, **kwargs):
    def wrapper(*f_args, **f_kwargs):
        new_kwargs = {**kwargs, **f_kwargs}
        return fn(*args, *f_args, **new_kwargs)
    return wrapper


def add_user(db, username):
    print(f"Adding user {username} to database {db}")

add_user_maindb = partial(add_user, "MainDB")
add_user_tempdb = partial(add_user, "TempDB")

add_user_maindb("alice")
add_user_maindb("bob")
add_user_maindb("charlie")

add_user_tempdb("eve")
add_user_tempdb("mallory")

Adding user alice to database MainDB
Adding user bob to database MainDB
Adding user charlie to database MainDB
Adding user eve to database TempDB
Adding user mallory to database TempDB


In [83]:
from functools import partial

def add_user(db, username):
    print(f"Adding user {username} to database {db}")

add_user_maindb = partial(add_user, "MainDB")
add_user_tempdb = partial(add_user, "TempDB")

add_user_maindb("alice")
add_user_maindb("bob")
add_user_maindb("charlie")

add_user_tempdb("eve")
add_user_tempdb("mallory")

Adding user alice to database MainDB
Adding user bob to database MainDB
Adding user charlie to database MainDB
Adding user eve to database TempDB
Adding user mallory to database TempDB


In [ ]:
from functools import singledispatch

@singledispatch
def process_data(data):
    #if isinstance(data, str):
    #    print(f"Processing string data: {data}")
    #elif isinstance(data, int):
    #    print(f"Processing integer data: {data}")
    #elif isinstance(data, list):
    #    print(f"Processing list data: {data}")
    print(f"Default Processor for data of type {type(data)}: {data}")

@process_data.register
def _(data: str):
    print(f"Processing string data: {data}")

from typing import Union

@process_data.register
def _(data: Union[int, float]):
    print(f"Processing integer data: {data}")

#@process_data.register(int)
#@process_data.register(float)
#def _(data):
#    print(f"Processing numeric data: {data}")

class User: pass

@process_data.register
def _(data: User):
    print(f"Processing User object: {data}")


@process_data.register
def _(data: list | tuple):
    print(f"Processing list/tuple data: {data}")

process_data("Hello, World!")
process_data(100)

u = User()
process_data(u)

process_data(1.2)

process_data([1, 2, 3])

Processing string data: Hello, World!
Processing numeric data: 100
Processing User object: <__main__.User object at 0x103a60770>
Processing numeric data: 1.2
Processing list/tuple data: [1, 2, 3]


In [92]:
def square(x):
    return x * x

square([44, 55, 66, 77])

TypeError: can't multiply sequence by non-int of type 'list'

In [95]:
from functools import singledispatch

@singledispatch
def square(x):
    return x * x

from typing import Union

@square.register
#def _(x: list | tuple):
def _(x: Union[list, tuple]):
    return [item * item for item in x]


print(square([44, 55, 66, 77]))
print(square(11))

[1936, 3025, 4356, 5929]
121



#### Capstone Project: `LogAnalyzer`

Implement a LogAnalyzer framework that has 3 distinct layers:

1. **The Ingestion Layer (Creational & Structural)**

This layer should be responsible for parsing logs passed as parameter, applying regexp capture filters (also passed as parameter) and store the filtered records into either CSV, JSON or SQLite table (based on parameters). This should also support mechanisms to pipeline data directly to Pandas DataFrame.
Things to keep in mind: timestamp formats in logs can vary, implement mechanisms to standardize them. Convert numerical values to integers/float - as this allows easier reduction/aggregration functions in the processing layer.

2. **The Processing Layer (Behavioral & Data Model)**

This layer should be responsible to provide mechanisms to filter, analyze and summarize data using various analysis engines.

3. **The Orchestration Layer (Front-end)**

This layer should provide a simple API mechanisms to ingest logs via the ingestion layers, apply record filters, run analysis and generate report that can be displayed / plotted using various front-end mechanisms.

#### Implementation approach:
 * **Stage 1** - Implement a simple code to parse logs and store into CSV, JSON and SQLite3 tables (using sshd_minimal.log first, then webserver access log).
 * **Stage 2** - Identify common patterns while processing different logs. Identify parameters that could be passed via configuration or arguments. Implement simple analysis strategies. Refactor the code to add extensibility and dynamicity for adding new log formats, filters and analysis.
 * **Stage 3** - Identify design patterns that could be applied in the code-base, refactor code to implement them.
 * **Stage 4** - Review code and check for redundancy and complexity, trim down features that are not required.

Test Link for the log files: https://files.chandrashekar.info/logs.zip


In [98]:
ins = open("project/sshd_minimal.log")
ins

<_io.TextIOWrapper name='project/sshd_minimal.log' mode='r' encoding='UTF-8'>

In [107]:
for line in ins:
    if "Failed password" in line:
        print(line.strip())
        break


Mar 05 19:34:57 moonranger sshd[3751843]: Failed password for invalid user ranjeet from 45.80.64.246 port 52676 ssh2


In [112]:
import re

regex = r'''
^(?P<timestamp>
 \w{3}    # Match month abbreviation
 \s       # Followed by a space
 \d{1,2}  # Day of the month (1 or 2 digits)
 \s\d{2}:\d{2}:\d{2} # Time in HH:MM:SS format
)
.+           # Skip to the relevant part
Failed       # Look for 'Failed password for invalid user' or 'Failed password for'
\s
password
\s
for
\s
(invalid \s user \s)?
(?P<username>\S+?)     # Extract username
\s from \s
(?P<ipaddr>\S+)        # Extract IP address
'''

pattern = re.compile(regex, re.VERBOSE)
if match := pattern.search(line):
    print(match.groupdict())

tuple(pattern.groupindex.keys())


{'timestamp': 'Mar 05 19:34:57', 'username': 'ranjeet', 'ipaddr': '45.80.64.246'}


('timestamp', 'username', 'ipaddr')

In [ ]:
# For webserver access logs: fields -> timestamp, client_ip, request_method, resource, response_code, bytes_transferred


In [113]:
def testfn():
    for i in range(10):
        yield i*i 

for val in testfn():
    print(val)

0
1
4
9
16
25
36
49
64
81


In [118]:
def accumlate():
    total = 0
    while True:
        value = yield
        if value is None:
            break
        total += value
        yield total
    return total


acc = accumlate()

data = [10, 20, 30, 40, None]

for total, value in zip(acc, data):
    acc.send(value)
    print(f"Sent: {value}, Accumulated Total: {total}")
        


Sent: 10, Accumulated Total: None
Sent: 20, Accumulated Total: None
Sent: 30, Accumulated Total: None
Sent: 40, Accumulated Total: None


StopIteration: 100

In [121]:
time_str = "Mar 10 14:23:45"
from datetime import datetime
dt = datetime.strptime(time_str, "%b %d %H:%M:%S")
print(dt)
dt.replace(year=2024).isoformat()

1900-03-10 14:23:45


'2024-03-10T14:23:45'

#### Further reading / reference

- Raymond Hettinger's Playlist: https://www.youtube.com/playlist?list=PLRVdut2KPAguz3xcd22i_o_onnmDKj3MA
- https://refactoring.guru/design-patterns/python
- Jeff Knupp's Videos: https://www.youtube.com/playlist?list=PLHb-tUHjtzRCMeC5IjbJOjNLiJ28982NJ
- Alex Martelli's Video: https://www.youtube.com/watch?v=LeuChRCByZc 


#### Contact details:
   Email: training@chandrashekar.info / chandra@slashprog.com
   
   https://www.chandrashekar.info/contact
   
